# Matched-training robustness study at 130M parameters (Kaggle)

Runs `scripts/scale_study.py` on Kaggle's GPUs. Same study as the Colab notebook; everything is fine-tuned again here with one setting, so all arms stay matched.

**Before running, in the right-hand panel (Settings):**
* Accelerator: **GPU T4 x2** (two arms run at once, one per GPU). P100 also works, one arm at a time. The TPU cannot be used.
* Internet: **On** (needs a phone-verified Kaggle account). The code, the checkpoint and WikiText-2 are downloaded.

**Recommended way to run:** *Save Version → Save & Run All (Commit)*. Kaggle then runs the notebook in the background for up to 12 hours and keeps the files in `/kaggle/working` as the version's **Output**. An interactive session stops when the browser tab is idle.

The notebook stops starting new runs after `BUDGET_HOURS`, so it finishes inside the 12-hour limit and the output is saved. To continue in a second session, add the finished version's output to the new notebook (*Add Input → Your Work → this notebook*) and run again: finished runs are skipped.

No token is needed.

In [ ]:
import os, time, glob, shutil, subprocess, torch
T0 = time.time()
BUDGET_HOURS = 9.0          # no new run starts after this; a 12-hour session then has time to finish and save
assert torch.cuda.is_available(), "No GPU: Settings > Accelerator > GPU T4 x2"
n_gpu = torch.cuda.device_count(); mem = torch.cuda.get_device_properties(0).total_memory / 2**30
PRESET = "t4" if mem < 20 else ("l4" if mem < 30 else "a100")
print(f"{n_gpu} x {torch.cuda.get_device_name(0)} ({mem:.0f} GB) -> preset {PRESET}, {n_gpu} run(s) at a time")
OUT = "/kaggle/working/lipmamba_scale_study"; os.makedirs(OUT + "/main", exist_ok=True)
# continue from an earlier version's output, if it was added as an input
for f in glob.glob("/kaggle/input/**/lipmamba_scale_study/main/*_s[0-9].json", recursive=True):
    if not os.path.exists(f"{OUT}/main/{os.path.basename(f)}"):
        shutil.copy(f, OUT + "/main/"); print("restored", os.path.basename(f))
os.environ.update(OUT=OUT, PRESET=PRESET, JOBS=str(n_gpu))

## 1. Code and checks (about 3 minutes)

In [ ]:
%cd /kaggle/working
!rm -rf CV && git clone -q --depth 1 -b claude/implement-lipmamba-model-SHbBY https://github.com/rogerpanel/CV.git
%cd /kaggle/working/CV/lipmamba
!pip -q install transformers datasets pytest
os.environ["PYTHONPATH"] = "/kaggle/working/CV/lipmamba/src"
!git log --oneline -1
!python -m pytest -q -o addopts="" tests/test_pretrained.py tests/test_scale_study.py tests/test_fast_scan.py 2>&1 | tail -1

## 2. The study

In order of importance: (a) the five core arms for three seeds, (b) four single-constraint ablations, (c) the paper's constants. Progress lines appear every five minutes; full logs are in `lipmamba_scale_study/main/logs`.

In [ ]:
def left():
    return max(0.01, BUDGET_HOURS - (time.time() - T0) / 3600)
base = f"python scripts/scale_study.py all --dir {OUT}/main --jobs {n_gpu}"
!{base} --seeds 0 1 2 --time-budget-hours {left():.2f} -- --preset {PRESET}
!{base} --seeds 0 --models lipcal_no_sn lipcal_no_clamp lipcal_free_A lipcal_no_clip --time-budget-hours {left():.2f} -- --preset {PRESET}
!{base} --seeds 0 1 2 --models lip lip_at --time-budget-hours {left():.2f} -- --preset {PRESET}

## 3. Summary and results file
Download `lipmamba_scale_results.zip` from the Output tab (or the file browser on the right) and upload it to the chat.

In [ ]:
!python scripts/scale_study.py aggregate --dir $OUT/main
!rm -f $OUT/main/*.ckpt.pt
!cd $OUT && zip -qr /kaggle/working/lipmamba_scale_results.zip main -x '*.pt'
!rm -rf /kaggle/working/CV
!ls -la /kaggle/working